In [2]:
from sage.all import *
from sage.combinat.sf.sf import SymmetricFunctions
from sage.stats.distributions.discrete_gaussian_integer import DiscreteGaussianDistributionIntegerSampler
from sage.stats.distributions.discrete_gaussian_lattice import DiscreteGaussianDistributionLatticeSampler
from random import randint
from random import Random
from fpylll import *
import hashlib
import statistics

import time

class CUFE_scheme():
    def __init__(self, m, n, q, K, security_parameter, sigma, verbosity=True):
        self.m = m
        self.n = n
        self.q = q
        self.K = K
        self.G = 0
        self.C = 5
        self.security_parameter = security_parameter
        self.rq = IntegerModRing(q)
        self.rho = 2
        self.rho_2 = 40
        self.mu = 0.5
        self.sigma = 2
        self.tau = 5

        self.verbose = verbosity

        if (self.verbose):
            print(f"On Gauss. parameters: sigma ({self.sigma}), rho ({self.rho}) and mu ({self.mu})")

        # Initialise these BEFORE setup so trap_gen can set them
        self.w = 0
        self.trap_norm = 0
        self.R = 0
        self.R0 = 0

        self.s = 0
        # Keys — trap_gen will correctly set self.R, self.w, self.G
        mpk, msk = self.setup(security_parameter)
        self.mpk = mpk
        self.msk = msk

        # DO NOT reinitialise self.w or self.R here

        self.scale = math.floor(self.q // self.K)

        if (self.verbose):
            print(f"Maximum K-scale (for bounded inner product results): {self.K}")

        self.ciphertexts = []
        self.updated = []

    def get_mpk(self):
        return self.mpk
    
    # Returns a vector modulo q that has been sampled from a Discrete Gaussian sampler
    def sample_dgauss_vector_mod_q(self, n, distro):
        D = DiscreteGaussianDistributionIntegerSampler(distro)

        return vector(ZZ, [D() for _ in range(n)])
    
    # Hash function using SHA-256 encryption
    def H(self, tag, rows, cols):
        digest = hashlib.sha256(tag.encode()).digest()
        seed = int.from_bytes(digest, 'big')
        rng = Random(seed)

        return matrix(Integers(self.q), rows, cols, 
                      [rng.randrange(self.q) for _ in range(rows*cols)])
    
    # Auxiliary helper functions calculating the hashes for each ciphertext
    # - Non-updated ciphertexts hash
    def H1(self, t): return self.H("H1"+t, self.n, self.m)

    # - Updated ciphertext hash
    def H2(self, t): return self.H("H2"+t, self.n, self.m)

    # - Target hash
    def H3(self, t): return self.H("H3"+t, self.n, self.m)

    # Creates a trapdoor using the approach of: https://people.csail.mit.edu/vinodv/6876-Fall2015/L16.pdf
    def trap_gen(self):
        # Parameter setup
        l = ceil(log(self.q, 2))
        w = self.n*l # Our w parameter

        if (self.verbose):
            print(f"Derived l and w parameters: {l} and {w} respectively")

        m_bar = self.m - w

        # Matrix of values 0 and 1 sampled at random
        R0 = matrix(ZZ, m_bar, w,[randint(0,1) for _ in range(m_bar*w)])
        
        I = identity_matrix(ZZ, w)

        # Forms our block matrix
        R = block_matrix([[R0], [I]])
        self.trap_norm = float(R.norm())
        #self.sigma = self.trap_norm * w * sqrt(log(self.m, 2))

        D = DiscreteGaussianDistributionIntegerSampler(self.sigma)

        # Forms a matrix mod q sampled across a discrete gaussian distribution
        A_bar = matrix(Integers(self.q), self.n, m_bar, 
                       [D() for _ in range(self.n*m_bar)])
        
        # Builds the gadget and sets it
        G = self.build_gadget()
        self.G = G

        self.s = self.trap_norm * w * (sqrt(math.log(self.n, 2)))
        A1 = (-A_bar*R0 + G) % self.q
        A = block_matrix([[A_bar, A1]])

        # Sanity check, ensures correct structure
        if (self.verbose):
            print(f"Sanity check, trapdoor structural correctness: {"PASS" if A*R == G else "FAIL"}")

        self.R = R
        self.w = w

        return A, R
    
    # Used in gadget inversion, converts values to bits and returns as a vector
    def bit_decomp(self, v):
        l = ceil(log(self.q,2))
        v_inv = int(v) % self.q
        return vector(ZZ, [v_inv >> i & 1 for i in range(l)])
    
    # Gadget building based on https://advancedcrypto.github.io/Lecture4a.pdf
    def build_gadget(self):
        I_n = identity_matrix(self.n)
        l = ceil(log(self.q, 2)) # required to ensure correct bit size
        g_pow_t = vector(ZZ, [2**i for i in range(l)]) # Gadget vector
        
        G = I_n.tensor_product(g_pow_t.row()) # Finds the Kronecker product
        return G
    
    # Uses vector-based bit decomposition and forms into matrix
    # Correctness can be proven via reconstruction, matrices always match!
    def gadget_inverse(self, v):
        return vector(ZZ, sum([list(self.bit_decomp(vi)) for vi in v], []))
    
    def gadget_test(self):
        q = self.q
        for _ in range(10):
            u = vector(Integers(q), [randint(0, q-1) for _ in range(self.n)])
            u_zz = vector(ZZ, [int(x) for x in u])
            g_inv = self.gadget_inverse(u_zz)
            recon = vector(Integers(q), self.G * g_inv)
            if recon != u:
                print("GADGET FAIL:", recon, "!=", u)
                return false
        print("Gadget test: PASSED!")
        return true
    
    # https://link.springer.com/chapter/10.1007/978-3-642-13190-5_28#preview
    def sample_left(self, M, u):
        D = DiscreteGaussianDistributionIntegerSampler(self.sigma)
        e_2 = vector(ZZ, [D() for _ in range(M.ncols())])  # length m

        # Keep arithmetic in Z_q
        u_q = vector(Integers(self.q), u)
        Me2_q = vector(Integers(self.q), M * e_2)
        y = u_q - Me2_q

        e_1 = self.sample_pre(y)
        return vector(ZZ, list(e_1) + list(e_2))

    def sample_left_as_matrix(self, M, H):
        cols = []
        for i in range(H.ncols()):
            u = vector(Integers(self.q), H.column(i))
            e = self.sample_left(M, u)
            cols.append(e)
        return Matrix(ZZ, cols).transpose()
    
    def sample_pre_as_matrix(self, M):
        cols = []
        for i in range(M.ncols()):
            u = vector(Integers(self.q), M.column(i))
            e = self.sample_pre(u)
            cols.append(e)
        return Matrix(ZZ, cols).transpose()
    
    def sample_pre(self, u):
        ZZ = IntegerRing()
        D = DiscreteGaussianDistributionIntegerSampler(self.s)

        # Solving Gz = u
        z = self.gadget_inverse(vector(ZZ, [int(x) for x in u]))

        y = vector(ZZ, [D() for _ in range(len(z))])

        x0 = self.R * z
        Ry = self.R * y

        Gy = self.G * y
        correction = self.R * self.gadget_inverse(Gy)
        z_kernel = Ry - correction

        e = x0 + z_kernel
        return e
    
    
    def setup(self, security_parameter):
        return self.trap_gen()
    
    def key_gen(self, tag, secret_vector):
        H_1 = self.H1(tag)
        H_2 = self.H2(tag)
        H_3 = self.H3(tag)

        Z_1 = self.sample_left_as_matrix(H_1, H_3)
        Z_2 = self.sample_left_as_matrix(H_2, H_3)

        y = vector(self.rq, secret_vector)

        sk_1 = Z_1 * y
        sk_2 = Z_2 * y

        return (sk_1, sk_2)
    
    def tok_gen(self, t, t_prim):
        B_t_1 = self.H1(t)
        B_t_prim_2 = self.H2(t_prim)
        D_t = self.H3(t)
        D_t_prim = self.H3(t_prim)

        D = DiscreteGaussianDistributionIntegerSampler(self.rho)

        Y_t_t_prim = matrix(ZZ, self.m, self.m, [D() for _ in range(self.m*self.m)])
        X_t_t_prim = self.sample_pre_as_matrix(B_t_prim_2 - B_t_1 * Y_t_t_prim)
        
        I_m = identity_matrix(ZZ, self.m)
        Z_mat = zero_matrix(ZZ, self.m, self.m)

        tok_t_t_prim_1 = block_matrix([[I_m, X_t_t_prim], [Z_mat, Y_t_t_prim]])
        tok_t_t_prim_2 = self.sample_left_as_matrix(B_t_1, D_t_prim - D_t)
        return (tok_t_t_prim_1, tok_t_t_prim_2)
    
    def encrypt(self, message, tag):
        B_t_1 = self.H1(tag)
        D_t = self.H3(tag)
        vec_s = vector(Integers(self.q), [randint(0, self.q-1) 
                                          for _ in range(self.n)])
        
        D_sigma = DiscreteGaussianDistributionIntegerSampler(self.sigma)
        D_mu = DiscreteGaussianDistributionIntegerSampler(self.mu)
        e_1 = vector(ZZ, [D_sigma() for _ in range(self.m)])
        e_2 = vector(ZZ, [D_sigma() for _ in range(self.m)])
        e_3 = vector(ZZ, [D_mu() for _ in range(self.m)])

        # sample S
        S = matrix(ZZ, self.m, self.m, [2*randint(0,1)-1 
                                        for _ in range(self.m*self.m)])
        H_t_1 = block_matrix([[self.mpk, B_t_1]])

        I_m = matrix.identity(self.m)
        f = block_matrix([[I_m], [S]]) * e_1

        ct_t_1_1 = H_t_1.T * vec_s + f
        scaled_message = message*self.scale
        ct_t_1_2 = D_t.T * vec_s + e_2 + e_3 + scaled_message

        return (ct_t_1_1, ct_t_1_2)
    
    def update(self, tok_part_1, tok_part_2, ct_t_1_1, ct_t_1_2, t_prim):
        if ((ct_t_1_1, ct_t_1_2) in self.updated):
            return "ERR", "Cannot make more than a single update"
        if (not ((ct_t_1_1, ct_t_1_2) in self.ciphertexts)):
            return "ERR", "Ciphertext invalid"
        
        D = DiscreteGaussianDistributionIntegerSampler(self.rho)
        
        B_t_prim_2 = self.H2(t_prim)
        D_t_prim = self.H3(t_prim)
        vec_r = vec_r = vector(self.rq, [randint(0, self.q-1) for _ in range(self.n)])
        vec_f_1 = vector(ZZ, [D() for _ in range(self.m*2)])
        vec_f_2 = vector(ZZ, [D() for _ in range(self.m)])
        
        H_t_prim_2 = block_matrix([[self.mpk, B_t_prim_2]])

        ct_t_prim_2_1 = tok_part_1.T*ct_t_1_1 + H_t_prim_2.T*vec_r + vec_f_1
        ct_t_prim_2_2 = ct_t_1_2 + tok_part_2.T*ct_t_1_1 + D_t_prim.T*vec_r + vec_f_2

        return (ct_t_prim_2_1, ct_t_prim_2_2)
    
    def decrypt(self, ct_t_l_1, ct_t_l_2, secret, sk_li):
        q = self.q

        # Deterministically select decryption key based on
        # membership to either fresh or updated ciphertext
        # arrays

        if ((ct_t_l_1, ct_t_l_2) in self.updated):
            sk_l = sk_li[1] # Key for updated ciphertexts
        else:
            sk_l = sk_li[0] # Key for fresh ciphertexts

        # Lift into integers mod q
        y_q = vector(Integers(q), secret)
        ct_1_q = vector(Integers(q), ct_t_l_1)
        ct_2_q = vector(Integers(q), ct_t_l_2)
        sk_q = vector(Integers(q), sk_l)

        mu_prime = y_q.dot_product(ct_2_q) - sk_q.dot_product(ct_1_q)
        val = int(mu_prime) % q # Lift from a ring into integers mod q

        return int(round(val/self.scale)) # Deviation 1

def test_update(CUFE, tag, new_tag, cipher_1, cipher_2, message_vector, secret_key):
        tok1, tok2 = CUFE.tok_gen(tag, new_tag)
        ucipher1, ucipher2 = CUFE.update(tok1, tok2, cipher_1, cipher_2, new_tag)

        if (ucipher1 == "ERR"):
            print("ERR: ", ucipher2)
            return False
        else:
            CUFE.ciphertexts.remove((cipher_1, cipher_2))
            CUFE.updated.append((ucipher1, ucipher2))

        function_key = CUFE.key_gen(new_tag, secret_key)
        result = CUFE.decrypt(ucipher1, ucipher2, secret_key, function_key)
        actual_result = message_vector.dot_product(secret_key)

        print(f"RETURNED: {result} VERSUS ACTUAL: {message_vector.dot_product(secret_key)}")

        print("\nASSESSING THE RESULT... please wait")
        error_value = round(abs(actual_result-result)/actual_result, 4)*100

        if (error_value != 0):
            print(f"Error occupies {error_value}% of the signal which {"is within limits" if (error_value <= ERR_THRESH) else "exceeds the limit"} of {ERR_THRESH}%")
            return error_value <= ERR_THRESH
        else:
            print(f"Error occupies ~0% of the signal, complete decryption success!")
            print("TOKEN UPDATES: PASS")
            return True
def test_decrypt(CUFE, secret_key, message_vector, tag, function_key):
    cipher1, cipher2 = CUFE.encrypt(message_vector, tag)
    CUFE.ciphertexts.append((cipher1, cipher2))

    result = CUFE.decrypt(cipher1, cipher2, secret_key, function_key)
    actual_result = message_vector.dot_product(secret_key)
    print(f"RETURNED: {result} VERSUS ACTUAL: {message_vector.dot_product(secret_key)}")

    print("\nASSESSING THE RESULT... please wait")
    if (actual_result == 0):
        error_value = round(abs(actual_result-result), 4)*100
    error_value = round(abs(actual_result-result)/actual_result, 4)*100

    success = False
        
    if (error_value != 0):
        success = True
        print(f"Error occupies {error_value}% of the signal which {"is within limits" if (error_value <= ERR_THRESH) else "exceeds the limit"} of {ERR_THRESH}%")
    else:
        success = True
        print(f"Error occupies ~0% of the signal, complete decryption success!")
    
    if (error_value > ERR_THRESH):
        success = False

    return CUFE, secret_key, message_vector, success
max_vector_value = 80
vector_size = 2 # n
lattice_size = 16 # m
security_bits = 256

modulus_exponent = int(lattice_size/vector_size)
modulus = 2**8

half = modulus / lattice_size
if (half < 2):
    P = 1
    V = 1
else:
    P = 1
    V = 1

if (not (lattice_size*P*V <= modulus)):
    raise Exception("Decrease n or increase m, cannot find appropriate P and V values!")

K = lattice_size*P*V

ERR_THRESH = 2.5

def single_run():
    secret_key = vector([randint(0, P) for _ in range(lattice_size)])
    message_vector = vector([randint(0, V) for _ in range(lattice_size)])

    CUFE = CUFE_scheme(lattice_size, vector_size, modulus, K, security_bits, sigma, True)
    tag = "diva"
    function_key = CUFE.key_gen(tag, secret_key)

    cipher1, cipher2 = CUFE.encrypt(message_vector, tag)
    CUFE.ciphertexts.append((cipher1, cipher2))

    result = CUFE.decrypt(cipher1, cipher2, secret_key, function_key)
    actual_result = message_vector.dot_product(secret_key)
    print(f"RETURNED: {result} VERSUS ACTUAL: {message_vector.dot_product(secret_key)}")

    print("\nASSESSING THE RESULT... please wait")
    error_value = round(abs(actual_result-result)/actual_result, 4)*100

    if (error_value != 0):
        print(f"Error occupies {error_value}% of the signal which {"is within limits" if (error_value <= ERR_THRESH) else "exceeds the limit"} of {ERR_THRESH}%")
    else:
        print(f"Error occupies ~0% of the signal, complete decryption success!")

    return CUFE, secret_key, message_vector

def multi_run(iterations=10):
    print(f"Running for {iterations} cycles and collating results")

    average_error = 0
    average_runtime = 0
    complete_successes = 0
    within_thresholds = 0
    outside_thresholds = 0

    runtime_results = []
    setup_results = []
    encrypt_results = []
    decrypt_results = []
    keygen_results = []
    error_results = []

    for i in range(iterations):
        setup_start = time.perf_counter()
        CUFE = CUFE_scheme(lattice_size, vector_size, modulus, K, security_bits, 2, False)
        setup_end = time.perf_counter()

        print(f"\rExecution {i} out of {iterations}", end='', flush=True)

        secret_key = vector([randint(0, P) for _ in range(lattice_size)])
        message_vector = vector([randint(0, V) for _ in range(lattice_size)])

        runstart = time.perf_counter()
        tag = "diva"

        keygen_start = time.perf_counter()
        function_key = CUFE.key_gen(tag, secret_key)
        keygen_end = time.perf_counter()

        encrypt_start = time.perf_counter()
        cipher1, cipher2 = CUFE.encrypt(message_vector, tag)
        CUFE.ciphertexts.append((cipher1, cipher2))
        encrypt_end = time.perf_counter()

        decrypt_start = time.perf_counter()
        result = CUFE.decrypt(cipher1, cipher2, secret_key, function_key)
        decrypt_end = time.perf_counter()

        runend = time.perf_counter()
        runtime = runend - runstart

        actual_result = message_vector.dot_product(secret_key)
        if (actual_result == 0):
            error_value = round(abs(actual_result - result), 4)*100
        else:
            error_value = round(abs(actual_result-result)/actual_result, 4)*100

        if (error_value == 0):
            complete_successes += 1
        elif (error_value <= ERR_THRESH):
            within_thresholds += 1
        else:
            outside_thresholds += 1
        
        runtime_results.append(runtime)
        setup_results.append(setup_end - setup_start)
        encrypt_results.append(encrypt_end - encrypt_start)
        decrypt_results.append(decrypt_end - decrypt_start)
        keygen_results.append(keygen_end - keygen_start)
        error_results.append(float(error_value))
    average_runtime = statistics.mean(runtime_results)
    runtime_std = statistics.stdev(runtime_results)

    average_setup = statistics.mean(setup_results)
    setup_std = statistics.stdev(setup_results)

    average_encrypt = statistics.mean(encrypt_results)
    encrypt_std = statistics.stdev(encrypt_results)

    average_decrypt = statistics.mean(decrypt_results)
    decrypt_std = statistics.stdev(decrypt_results)

    average_keygen = statistics.mean(keygen_results)
    keygen_std = statistics.stdev(keygen_results)

    average_error = statistics.mean(error_results)
    error_std = statistics.stdev(error_results)

    print(f"\nOut of the {iterations} runs:")
    print(f"{complete_successes} COMPLETE SUCCESSES")
    print(f"{within_thresholds} WITHIN THRESHOLD")
    print(f"{outside_thresholds} OUTSIDE THRESHOLD")

    print(f"\nRuntime: {average_runtime:.4f}s ± {runtime_std:.4f}s")
    print(f"Setup: {average_setup:.4f}s ± {setup_std:.4f}s")
    print(f"Encryption: {average_encrypt:.4f}s ± {encrypt_std:.4f}s")
    print(f"Decryption: {average_decrypt:.4f}s ± {decrypt_std:.4f}s")
    print(f"KeyGen: {average_keygen:.4f}s ± {keygen_std:.4f}s")

    print(f"\nError: {average_error:.4f}% ± {error_std:.4f}%")
    return CUFE, secret_key, message_vector

#for k in range(4, 8):
#    new_start = k+3
#    for j in range(new_start, 11):
#        lattice_size = 2**j
#        vector_size = 2**k
#        iters = 100
#        if (lattice_size > 128):
#            iters = 5

#        print(f"Running dimension size: {vector_size}x{lattice_size} (NxM) for modulus (q): {modulus}")
#        print(f"On max message (P): {P} and max secret (V): {V}")
#        multi_run(iters)

#CUFE, secret_key, message_vector = multi_run(10)
CUFE, secret_key, message_vector = single_run()

testing_enabled = False
if (testing_enabled):
    print(f"\n_____________________EXECUTING TEST SUITE:_____________________")
    print(f"Testing GADGET INVERSION, please wait...")
    u_test = vector(Integers(CUFE.q), [randint(0, CUFE.q-1) for _ in range(CUFE.n)])
    u_int = vector(ZZ, [int(x) for x in u_test])
    g_inv = CUFE.gadget_inverse(u_int)
    reconstructed = CUFE.G * g_inv
    print(f"Result: {"PASS" if vector(Integers(CUFE.q), reconstructed) == u_test else "FAIL"}")

    print(f"Testing TOKEN UPDATES, please wait...")
    result = test_update(CUFE, "diva", "icon", CUFE.ciphertexts[0][0], CUFE.ciphertexts[0][1], message_vector, secret_key)

    print(f"Testing TOKEN UPDATE on a mismatched ciphertext, please wait...")
    c_1, c_2 = CUFE.encrypt(message_vector, "New")

    CUFE.ciphertexts.append((c_1, c_2))
    result = test_update(CUFE, "Old", "None", c_1, c_2, message_vector, secret_key)
    print("SUCCESS") if not result else print("FAILURE")

    print(f"Trying to update an already updated ciphertext")
    result = test_update(CUFE, "icon", "icon2", CUFE.updated[0][0], CUFE.updated[0][1], message_vector, secret_key)
    print("SUCCESS") if not result else print("FAILURE")

    print(f"Trying to decrypt with an invalid functional key")
    function_key = CUFE.key_gen("othertag", secret_key)
    result = test_decrypt(CUFE, secret_key, message_vector, "diva", function_key)
    print("SUCCESS") if not result else print("FAILURE")



On Gauss. parameters: sigma (2), rho (2) and mu (0.5)
Derived l and w parameters: 8 and 16 respectively
Sanity check, trapdoor structural correctness: PASS
Maximum K-scale (for bounded inner product results): 16
RETURNED: 13 VERSUS ACTUAL: 4

ASSESSING THE RESULT... please wait
Error occupies 225.0% of the signal which exceeds the limit of 2.5%
